# Predicción de actividad por zona (1 a 7 días)

Cuaderno de experimentación con un CSV de prueba de abril (3 semanas). **La versión que corre en Artisa** está en `machine_learning/prediccion.py`: lee MariaDB (histórico + API en vivo), solo camiones del catálogo y días de Chile, se reentrena cada noche con el cierre diario y se ve en la web, debajo del Mapa de rutas.

Diferencias encontradas al auditar este cuaderno (y corregidas en la versión de Artisa):
- El early stopping se detuvo en **1 árbol** y el modelo **no superó** a la referencia ingenua (MAE 1,96 contra 1,92).
- `dia_idx` siempre queda fuera del rango entrenado al predecir: el árbol lo trata como constante y los 7 días salen casi iguales.
- La grilla depende del mínimo de cada CSV; en Artisa es fija (celdas de 0,01°).
- Los días se agrupan en UTC; en Artisa, de 00:00 a 00:00 hora de Chile.


## 1. Definir el problema y el objetivo (y)

Instala las librerías base (tablas y gráficos) que se usan en todo el notebook.

In [ ]:
%pip install pandas matplotlib

## 2. Explorar los datos (EDA)

Carga el CSV de GPS y hace una primera inspección (`info`/`describe`).

In [ ]:
import pandas as pd

df = pd.read_csv("datos_gps_3_semanas.csv"  # CSV de prueba (abril), no se sube a Git)
df.head()

In [ ]:
df.info()
df.describe()

## 3. Limpiar: duplicados, tipos, valores imposibles

eliminar duplicados

In [ ]:
cols_comparar = [
    "report_date",              # fecha y hora del reporte
    "asset_id", "asset_name", "event_name",
    "latitude", "longitude", "location",
    "speed", "cartography_limit_speed",
    "driver_name", "driver_last_name", "driver_document_number",
]

df_original = df.copy()   # respaldo para poder comparar

antes = len(df)
df = df.drop_duplicates(subset=cols_comparar, keep="first").reset_index(drop=True)
print(f"Filas: {antes:,} -> {len(df):,} (eliminadas: {antes - len(df):,})")

Porcentaje que se conservó
Reportes por vehículo, antes de limpiar
Lo mismo, después de limpiar

In [ ]:
print(f"Únicos: {len(df):,} de {len(df_original):,} ({len(df)/len(df_original):.1%})")
print(df_original.groupby("asset_id", observed=True).size().describe())
print(df.groupby("asset_id", observed=True).size().describe())

Cuenta cuántos reportes tiene cada vehículo y muestra los que tienen menos y los que tienen más.

In [ ]:
por_asset = df.groupby(["asset_id", "asset_name"], observed=True).size().sort_values()
print(por_asset.head(10))   # los de menos reportes
print(por_asset.tail(5))    # 

Revisa durante qué fechas reportaron los vehículos con pocos reportes.

In [ ]:
pocos = ["RYCV75", "RYCV76", "TFWK20", "RWSC78", "SWDL61"]
resumen = (df[df["asset_id"].isin(pocos)]
           .groupby("asset_id", observed=True)["report_date"]
           .agg(primero="min", ultimo="max", reportes="count"))
print(resumen)

print("Rango total:", df["report_date"].min(), "->", df["report_date"].max())

Convierte cada columna al tipo de dato que le corresponde, y ordena el DataFrame.



In [ ]:
# --- 3.2 Tipos ---
df["report_date"] = pd.to_datetime(df["report_date"])
df["speed"] = df["speed"].astype("int16")
df["cartography_limit_speed"] = df["cartography_limit_speed"].astype("int16")
for c in ["asset_id", "asset_name", "event_name"]:
    df[c] = df[c].astype("category")
df = df.sort_values(["asset_id", "report_date"]).reset_index(drop=True)
df.info()

eliminamos la columna de fecha de ingesta

In [ ]:
df = df.drop(columns=["ingestado_en"])
df.info()

Falta la parte 3.3. Ejecuta el diagnóstico de valores imposibles 

In [ ]:
print(df[["speed", "cartography_limit_speed", "latitude", "longitude"]].describe())
print("speed < 0:", (df["speed"] < 0).sum())
print("speed > 200:", (df["speed"] > 200).sum())
print("speed > límite:", (df["speed"] > df["cartography_limit_speed"]).sum())
print("lat fuera de Chile:", (~df["latitude"].between(-56, -17)).sum())
print("lon fuera de Chile:", (~df["longitude"].between(-76, -66)).sum())

Los números cuadran: 62.218 − 14.674 = 47.544. Es decir, la mayor parte (76 %) de los “excesos” no eran excesos, eran vehículos en movimiento en tramos sin límite cargado en la cartografía.

In [ ]:
sin_limite = df["cartography_limit_speed"] == 0
print("limite = 0:", sin_limite.sum(), f"({sin_limite.mean():.1%})")

exceso_real = (df["speed"] > df["cartography_limit_speed"]) & ~sin_limite
print("exceso con límite válido:", exceso_real.sum())

# ¿Los excesos reales son moderados o extremos?
exceso = (df["speed"] - df["cartography_limit_speed"])[exceso_real]
print(exceso.describe())

# ¿Velocidad > 0 sin límite? (se mueve sin cartografía)
print("se mueve sin límite:", (sin_limite & (df["speed"] > 0)).sum())

Qué hacer:

Excesos reales: conservarlos. No son errores, y pueden ser lo que quieres modelar.
Límite 0: convertirlo a faltante (NaN) para que el modelo no lo lea como “el límite es 0 km/h”. Los dejo como están, sin borrar filas.
speed: no se toca; no hay negativos ni valores imposibles.

In [ ]:
# Límite 0 = desconocido -> NaN (Int16 con mayúscula admite nulos)
df["cartography_limit_speed"] = (
    df["cartography_limit_speed"].replace(0, pd.NA).astype("Int16")
)

print("Límite faltante:", df["cartography_limit_speed"].isna().sum())

# Revisar los excesos más extremos antes de darlos por buenos
df["exceso"] = df["speed"] - df["cartography_limit_speed"]
print(df.loc[df["exceso"] > 50,
             ["asset_id", "report_date", "speed", "cartography_limit_speed", "location"]].head(10))

Resumen de la limpieza: filas antes/después, tipos de datos cambiados, memoria ahorrada.

In [ ]:
# 4) Reporte: qué se hizo y resultado antes vs después
filas_antes, filas_despues = len(df_original), len(df)
dup_antes = df_original.drop(columns="report_id").duplicated().sum()
dup_despues = df.drop(columns="report_id").duplicated().sum()

print("=== REPORTE DE LIMPIEZA ===\n")
print("1) Duplicados eliminados:", "SÍ" if dup_antes > 0 and dup_despues == 0 else "NO")
print(f"   Filas: {filas_antes:,} -> {filas_despues:,}  ({filas_antes - filas_despues:,} eliminadas)")
print(f"   Duplicados antes: {dup_antes:,} | después: {dup_despues:,}\n")

print("2) Tipos de datos cambiados:")
tipos = pd.DataFrame({"antes": df_original.dtypes.astype(str), "después": df.dtypes.astype(str)})
tipos["¿cambió?"] = tipos["antes"] != tipos["después"]
print(tipos)

mem_antes = df_original.memory_usage(deep=True).sum() / 1e6
mem_despues = df.memory_usage(deep=True).sum() / 1e6
print(f"\n3) Memoria: {mem_antes:.1f} MB -> {mem_despues:.1f} MB")
df.head()

## 4. Predicción de actividad por día (1 a 7 días, a elección)

Un solo modelo: predice cuánta actividad va a haber en cada zona del mapa, día por día, con el ciclo completo — **entrenamiento, validación, test, chequeo de overfitting/underfitting con early stopping, y comparación contra datos reales** — antes de mostrar el mapa de calor predicho.

**Sobre el horizonte:** el CSV tiene 3 semanas. Hoy se puede predecir de forma confiable de 1 a 7 días (ya validado más abajo). A medida que conectes la base de datos y se acumulen más meses, el mismo código (con la persistencia que se agrega al final) va a poder extenderse a semanas y meses, porque va a tener más historia para aprender el patrón.

In [ ]:
# ---- 1) Grilla diaria de actividad (conteo de reportes en movimiento por celda y dia) ----
import numpy as np
import pandas as pd

CELDA = 0.01
mov = df[df["speed"] > 2].copy()
lat0, lon0 = mov["latitude"].min(), mov["longitude"].min()
ny = int((mov["latitude"].max() - lat0) / CELDA) + 1
nx = int((mov["longitude"].max() - lon0) / CELDA) + 1

mov["dia"] = mov["report_date"].dt.date
mov["celda_i"] = ((mov["latitude"] - lat0) / CELDA).astype(int).clip(0, ny - 1)
mov["celda_j"] = ((mov["longitude"] - lon0) / CELDA).astype(int).clip(0, nx - 1)

dias = sorted(mov["dia"].unique())
print(f"Dias disponibles: {len(dias)} ({dias[0]} a {dias[-1]})")

grillas = {}
for dia, g in mov.groupby("dia"):
    grilla = np.zeros((ny, nx))
    np.add.at(grilla, (g["celda_i"].to_numpy(), g["celda_j"].to_numpy()), 1)
    grillas[dia] = grilla

# Solo nos interesan las celdas que alguna vez tuvieron actividad (el resto del mapa
# nunca se visita, no tiene sentido pedirle al modelo que aprenda sobre celdas vacias)
total_historico = np.sum(list(grillas.values()), axis=0)
celdas_activas = list(zip(*np.where(total_historico > 0)))
print(f"Celdas activas (con historial real): {len(celdas_activas)} de {ny * nx} celdas totales")

In [ ]:
# ---- 2) Armar la tabla (dia, celda) -> cantidad, con features sin fuga de datos ----
filas = []
for idx_dia, dia in enumerate(dias):
    grilla_dia = grillas[dia]
    for (i, j) in celdas_activas:
        filas.append({
            "dia": dia,
            "dia_idx": idx_dia,
            "dia_semana": pd.Timestamp(dia).weekday(),
            "celda_i": i, "celda_j": j,
            "lat_centro": lat0 + (i + 0.5) * CELDA,
            "lon_centro": lon0 + (j + 0.5) * CELDA,
            "conteo": grilla_dia[i, j],
        })

tabla = pd.DataFrame(filas).sort_values(["celda_i", "celda_j", "dia_idx"]).reset_index(drop=True)

# lag1 y media historica: SOLO miran dias anteriores al que se esta prediciendo (sin fuga)
g_celda = tabla.groupby(["celda_i", "celda_j"])["conteo"]
tabla["lag1"] = g_celda.shift(1)
tabla["media_historica"] = g_celda.transform(lambda s: s.shift(1).expanding().mean())

# El primer dia de cada celda no tiene historia previa: se descarta
tabla = tabla.dropna(subset=["lag1", "media_historica"]).reset_index(drop=True)

# Objetivo relativo (no el conteo directo): un arbol de decision no es bueno
# "copiando" casi exacto un numero continuo (media_historica); en cambio, aprender
# el ajuste relativo a lo habitual de esa celda es un problema mucho mas facil
# con pocos dias de historia. +1 en numerador/denominador evita dividir por cero.
tabla["objetivo_ratio"] = (tabla["conteo"] + 1) / (tabla["media_historica"] + 1)

print(f"Filas (dia x celda) para entrenar: {len(tabla):,}")

**Separación en train / validación / test**, por orden de tiempo (nunca al azar — es una serie de tiempo): train para entrenar, validación para el early stopping (elegir cuándo parar de entrenar), y test para la evaluación final, que el modelo no toca para nada durante el entrenamiento.

In [ ]:
# lat/lon no entran como feature: con pocos dias actuan casi como un ID de celda y
# el arbol memoriza en vez de aprender el patron temporal (ya lo capta media_historica)
feats = ["dia_semana", "dia_idx", "lag1", "media_historica"]

dias_idx_disponibles = sorted(tabla["dia_idx"].unique())
n = len(dias_idx_disponibles)
corte_train = dias_idx_disponibles[int(n * 0.60)]
corte_val = dias_idx_disponibles[int(n * 0.80)]

train = tabla[tabla["dia_idx"] <= corte_train]
val = tabla[(tabla["dia_idx"] > corte_train) & (tabla["dia_idx"] <= corte_val)]
test = tabla[tabla["dia_idx"] > corte_val]

print(f"Dias en train: hasta indice {corte_train} ({train['dia'].nunique()} dias, {len(train):,} filas)")
print(f"Dias en validacion: hasta indice {corte_val} ({val['dia'].nunique()} dias, {len(val):,} filas)")
print(f"Dias en test: resto ({test['dia'].nunique()} dias, {len(test):,} filas)")

### Entrenamiento con early stopping

Se entrena mirando el error en **validación** en cada ronda, y se detiene solo cuando deja de mejorar — así no se entrena ni de más (sobreajuste) ni de menos (subajuste).

In [ ]:
import lightgbm as lgb

# regularizado, pero sin feature_fraction: con solo 4 variables, submuestrearlas
# dejaba afuera a "media_historica" (la mas importante) en varios arboles
params = {"objective": "regression", "metric": "l2", "learning_rate": 0.05,
          "num_leaves": 15, "min_data_in_leaf": 30, "seed": 42, "verbose": -1}

ds_train = lgb.Dataset(train[feats], label=train["objetivo_ratio"])
ds_val = lgb.Dataset(val[feats], label=val["objetivo_ratio"], reference=ds_train)

modelo_dia = lgb.train(params, ds_train, num_boost_round=2000, valid_sets=[ds_val],
                       callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)])

print(f"Rondas optimas encontradas por early stopping: {modelo_dia.best_iteration}")

### Diagnóstico de overfitting / underfitting

Comparamos el error en los tres conjuntos. Si train es mucho mejor que test → sobreajuste. Si los tres son malos → subajuste. Si train/val/test son parecidos y buenos → el modelo generaliza bien.

In [ ]:
def predecir_conteo(conjunto):
    """Predice el ratio y lo vuelve a convertir a conteo esperado."""
    pred_ratio = modelo_dia.predict(conjunto[feats])
    pred_conteo = pred_ratio * (conjunto["media_historica"] + 1) - 1
    return np.clip(pred_conteo, 0, None)


def error_mae(conjunto):
    pred = predecir_conteo(conjunto)
    return float(np.abs(pred - conjunto["conteo"]).mean())


mae_train = error_mae(train)
mae_val = error_mae(val)
mae_test = error_mae(test)

print("Error absoluto medio (MAE), en reportes por celda-dia:")
print(f"  Train:      {mae_train:.2f}")
print(f"  Validacion: {mae_val:.2f}")
print(f"  Test:       {mae_test:.2f}")

brecha = mae_test - mae_train
baseline_ingenuo = float(np.abs(test["media_historica"] - test["conteo"]).mean())
print(f"\nReferencia ingenua (predecir siempre el promedio historico de esa celda): {baseline_ingenuo:.2f}")

if brecha > mae_train:
    print("-> Brecha grande entre train y test: posible SOBREAJUSTE.")
elif mae_test >= baseline_ingenuo:
    print("-> El modelo no supera a la referencia ingenua: posible SUBAJUSTE.")
else:
    print("-> Train, validacion y test son parecidos, y el modelo supera a la referencia: buen ajuste.")

### Comparación contra datos reales

Para cada día de test (que el modelo nunca vio), comparamos el mapa predicho contra el mapa real de ese día — numéricamente (correlación) y visualmente (lado a lado).

In [ ]:
# Comparacion numerica, dia por dia de test
resultados_test = []
for dia in sorted(test["dia"].unique()):
    sub = test[test["dia"] == dia]
    pred = predecir_conteo(sub)
    real = sub["conteo"].to_numpy()
    corr = float(np.corrcoef(pred, real)[0, 1]) if len(pred) > 1 else float("nan")
    mae_dia = float(np.abs(pred - real).mean())
    resultados_test.append({"dia": dia, "mae": round(mae_dia, 2), "correlacion": round(corr, 3)})

resultados_test_df = pd.DataFrame(resultados_test)
print(resultados_test_df)
print(f"\nCorrelacion promedio prediccion vs realidad (dias de test, nunca vistos): "
      f"{resultados_test_df['correlacion'].mean():.3f}")

In [ ]:
# Comparacion visual: un dia de test, mapa predicho vs mapa real, lado a lado
import matplotlib.pyplot as plt

dia_ejemplo = sorted(test["dia"].unique())[0]
sub = test[test["dia"] == dia_ejemplo]

grilla_pred = np.zeros((ny, nx))
grilla_real = np.zeros((ny, nx))
pred_vals = predecir_conteo(sub)
for (i, j), p, r in zip(sub[["celda_i", "celda_j"]].to_numpy(), pred_vals, sub["conteo"]):
    grilla_pred[i, j] = max(p, 0)
    grilla_real[i, j] = r

ext = [lon0, lon0 + nx * CELDA, lat0, lat0 + ny * CELDA]
fig, axes = plt.subplots(1, 2, figsize=(14, 7))
axes[0].imshow(np.log1p(grilla_pred), origin="lower", extent=ext, cmap="hot", aspect="auto")
axes[0].set_title(f"PREDICHO — {dia_ejemplo}")
axes[1].imshow(np.log1p(grilla_real), origin="lower", extent=ext, cmap="hot", aspect="auto")
axes[1].set_title(f"REAL — {dia_ejemplo}")
for ax in axes:
    ax.set_xlabel("longitude"); ax.set_ylabel("latitude")
plt.tight_layout()
plt.show()

### Persistencia (para que mañana sume datos nuevos, no reentrene de cero)

Igual que en las otras secciones: guarda el modelo y recuerda hasta qué día ya entrenó, para no duplicar el mismo mes si corrés esto de nuevo.

In [ ]:
import json
from pathlib import Path

RUTA_MODELO_DIA = Path("modelo_dia.txt")
RUTA_MARCA_DIA = Path("modelo_dia_marca.json")


def _leer_marca():
    if RUTA_MARCA_DIA.exists():
        with open(RUTA_MARCA_DIA, "r", encoding="utf-8") as f:
            return pd.Timestamp(json.load(f)["hasta"]).date()
    return None


def _guardar_marca(dia):
    with open(RUTA_MARCA_DIA, "w", encoding="utf-8") as f:
        json.dump({"hasta": str(dia)}, f)


marca_previa = _leer_marca()
if marca_previa is not None and dias[-1] <= marca_previa:
    print(f"Sin dias mas nuevos que la ultima corrida ({marca_previa}) -> NO se reentrena.")
    modelo_dia = lgb.Booster(model_file=str(RUTA_MODELO_DIA))
else:
    modelo_dia.save_model(str(RUTA_MODELO_DIA))
    _guardar_marca(dias[-1])
    print(f"Modelo de actividad diaria guardado, con datos hasta {dias[-1]}.")

### Predicción final: 1 a 7 días a elección

Elegí el horizonte en `N_DIAS` (1 = mañana, 7 = una semana). El modelo predice día por día, usando su propia predicción del día anterior como `lag1` del siguiente (igual que un pronóstico del tiempo: hoy predice mañana, y con eso predice pasado mañana).

In [ ]:
def predecir_n_dias(n_dias):
    """Predice la grilla de actividad para cada uno de los proximos n_dias (1 a 7 recomendado;
    mas adelante, con mas meses de historia acumulada, se puede pedir mas dias con confianza)."""
    lag_actual = tabla.groupby(["celda_i", "celda_j"])["conteo"].last().to_dict()
    media_hist = tabla.groupby(["celda_i", "celda_j"])["conteo"].mean().to_dict()
    ultimo_dia_idx = tabla["dia_idx"].max()

    grillas_pred = []
    for paso in range(1, n_dias + 1):
        dia_idx_pred = ultimo_dia_idx + paso
        fecha_pred = pd.Timestamp(dias[-1]) + pd.Timedelta(days=paso)
        filas_pred = pd.DataFrame([{
            "dia_semana": fecha_pred.weekday(), "dia_idx": dia_idx_pred,
            "lat_centro": lat0 + (i + 0.5) * CELDA, "lon_centro": lon0 + (j + 0.5) * CELDA,
            "lag1": lag_actual[(i, j)], "media_historica": media_hist[(i, j)],
            "celda_i": i, "celda_j": j,
        } for (i, j) in celdas_activas])

        pred_ratio = modelo_dia.predict(filas_pred[feats])
        pred = (pred_ratio * (filas_pred["media_historica"] + 1) - 1).clip(lower=0)
        grilla = np.zeros((ny, nx))
        for (i, j), p in zip(filas_pred[["celda_i", "celda_j"]].to_numpy(), pred):
            grilla[i, j] = p
            lag_actual[(i, j)] = p   # la prediccion de hoy es el lag de mañana
        grillas_pred.append(grilla)

    return grillas_pred


# <<< Elegi el horizonte aca: de 1 (mañana) a 7 (una semana) >>>
N_DIAS = 7
grillas_predichas = predecir_n_dias(N_DIAS)
grilla_total = np.sum(grillas_predichas, axis=0)
print(f"Prediccion para los proximos {N_DIAS} dias, desde {pd.Timestamp(dias[-1]).date() + pd.Timedelta(days=1)}")
print(f"Total de actividad predicha: {grilla_total.sum():,.0f} reportes")

In [ ]:
# Mapa de calor PREDICHO (version estatica)
ext = [lon0, lon0 + nx * CELDA, lat0, lat0 + ny * CELDA]

plt.figure(figsize=(8, 8))
plt.imshow(np.log1p(grilla_total), origin="lower", extent=ext, cmap="hot", aspect="auto")
plt.colorbar(label=f"log(actividad predicha en {N_DIAS} dias)")
plt.xlabel("longitude"); plt.ylabel("latitude")
plt.title(f"Mapa de calor PREDICHO: proximos {N_DIAS} dias")
plt.show()

In [ ]:
# Mismo mapa, version interactiva (folium)
import folium
from folium.plugins import HeatMap

puntos_calor = []
it = np.nditer(grilla_total, flags=["multi_index"])
for valor in it:
    if valor > 0:
        i, j = it.multi_index
        puntos_calor.append([lat0 + (i + 0.5) * CELDA, lon0 + (j + 0.5) * CELDA, float(valor)])

mp = folium.Map(location=[mov["latitude"].mean(), mov["longitude"].mean()], zoom_start=7)
HeatMap(puntos_calor, radius=8, blur=10, min_opacity=0.3).add_to(mp)
mp.save("mapa_calor_prediccion_dias.html")
mp

### Mapa de datos REALES (histórico completo), para comparar

El mismo estilo de mapa que el predicho de arriba, pero con lo que **efectivamente ocurrió** en las 3 semanas de datos — para comparar visualmente si se parecen.

In [ ]:
plt.figure(figsize=(8, 8))
plt.imshow(np.log1p(total_historico), origin="lower", extent=ext, cmap="hot", aspect="auto")
plt.colorbar(label="log(actividad real historica)")
plt.xlabel("longitude"); plt.ylabel("latitude")
plt.title("Mapa de calor REAL (histórico, 3 semanas)")
plt.show()